In [1]:
# imports
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.colors import TwoSlopeNorm
import rasterio
from rasterio.mask import mask as rio_mask
import geopandas as gpd
from pyproj import Transformer

In [ ]:
# ---
# Config -- edit paths and options
# ---

DATA = Path("../data")

TRUE_FULL   = DATA / "Canwell_FUllDomain_Diff3m.tif"
PRED_RASTER = DATA / "Canwell_gnn_northslopepreds_clipped.tif"
SLOPE_GPKG  = DATA / "CanwellNorthSlope_clean_utm.gpkg"

OUT_DIR = Path(".")
FLIP_SIGN = True         # orig (+) = loss
CLIP_PCT  = 98           # symmetric percentile for color limits
SYMMETRIC = True         # True: zero-centered symmetric; False: zero-centered assym.
CMAP      = "RdBu_r"     # blue = negative (loss), red = positive (gain) after _r
DPI       = 300
DATE_LABEL = "2000\u20132025"  # for the color label

In [ ]:
# ----- 

In [ ]:
def load_clipped(raster_path, gpkg_path=None):
    """read band 1, optionally clip to a polygon, return (array, transform, crs, bounds).
        Nodata and -9999 sentinels are converted to NaN."""
    with rasterio.open(raster_path) as src:
        if gpkg_path is not None:
            gdf = gpd.read_file(gpkg_path)
            if gdf.crs != src.crs:
                gdf = gdf.to_crs(src.crs)
            geoms = [g.__geo_interface__ for g in gdf.geometry]
            arr, transform = rio_mask(src, geoms, crop=True, filled=True, nodata=np.nan)
            arr = arr[0]
        else:
            arr = src.read(1).astype("float64")
            transform = src.transform
        crs = src.crs
        nodata = src.nodata
    arr = arr.astype("float64")
    if nodata is not None and not np.isdata(nodata):
        arr[arr == nodata] = np.nan
    arr[arr == -9999.0] = np.nan
    # recompute bounds from the (possibly cropped) transform + shape
    h, w = arr.shape
    left, top = transform * (0, 0)
    right, bottom = transform * (w, h)
    bounds = (left, bottom, right, top)
    return arr, transform, crs, bounds